In [1]:
import mlflow

# The set_experiment API creates a new experiment if it doesn't exist.
mlflow.set_experiment("Hyperparameter Tuning Experiment")

/home/fucci/repos/jupyter-mlflow-iris/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='/home/fucci/repos/jupyter-mlflow-iris/mlruns/3', creation_time=1782963880778, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1782963880778, lifecycle_stage='active', name='Hyperparameter Tuning Experiment', tags={}, trace_location=None, workspace='default'>

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=0)

In [3]:
import mlflow
import optuna
import sklearn


def objective(trial):
    # Setting nested=True will create a child run under the parent run.
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:
        rf_max_depth = trial.suggest_int("rf_max_depth", 2, 32)
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300, step=10)
        rf_max_features = trial.suggest_float("rf_max_features", 0.2, 1.0)
        params = {
            "max_depth": rf_max_depth,
            "n_estimators": rf_n_estimators,
            "max_features": rf_max_features,
        }
        # Log current trial's parameters
        mlflow.log_params(params)

        regressor_obj = sklearn.ensemble.RandomForestRegressor(**params)
        regressor_obj.fit(X_train, y_train)

        y_pred = regressor_obj.predict(X_val)
        error = sklearn.metrics.mean_squared_error(y_val, y_pred)
        # Log current trial's error metric
        mlflow.log_metrics({"error": error})

        # Log the model file
        mlflow.sklearn.log_model(regressor_obj, name="model")
        # Make it easy to retrieve the best-performing child run later
        trial.set_user_attr("run_id", child_run.info.run_id)
        return error

In [4]:
# Create a parent run that contains all child runs for different trials
with mlflow.start_run(run_name="study") as run:
    # Log the experiment settings
    n_trials = 30
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    # Log the best trial and its run ID
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metrics({"best_error": study.best_value})
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)

[I 2026-07-01 21:07:13,988] A new study created in memory with name: no-name-455019bb-15f3-42b9-985d-15596cf12617
[I 2026-07-01 21:07:34,476] Trial 0 finished with value: 0.51926864044188 and parameters: {'rf_max_depth': 4, 'rf_n_estimators': 60, 'rf_max_features': 0.8311132489795174}. Best is trial 0 with value: 0.51926864044188.
[I 2026-07-01 21:08:05,952] Trial 1 finished with value: 0.2568693296247485 and parameters: {'rf_max_depth': 22, 'rf_n_estimators': 270, 'rf_max_features': 0.6821917548800531}. Best is trial 1 with value: 0.2568693296247485.
[I 2026-07-01 21:08:42,347] Trial 2 finished with value: 0.24397293686677457 and parameters: {'rf_max_depth': 25, 'rf_n_estimators': 300, 'rf_max_features': 0.49459238183260096}. Best is trial 2 with value: 0.24397293686677457.
[I 2026-07-01 21:08:54,978] Trial 3 finished with value: 0.7136615326045985 and parameters: {'rf_max_depth': 2, 'rf_n_estimators': 100, 'rf_max_features': 0.7113310783873592}. Best is trial 2 with value: 0.24397293

In [6]:
# Register the best model using the model URI
run_id = study.best_trial.user_attrs.get("run_id")
print(f"Best trial run ID: {run_id}")


mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name="housing-price-predictor",
)
# > Successfully registered model 'housing-price-predictor'.
# > Created version '1' of model 'housing-price-predictor'.

Successfully registered model 'housing-price-predictor'.
2026/07/01 21:26:37 WARNING mlflow.tracking._model_registry.fluent: Run with id 25c7141d97e94af783847f43328b7b8e has no artifacts at artifact path 'model', registering model based on models:/m-5f4d554584d04dbd84c7e5c7514fc97c instead


Best trial run ID: 25c7141d97e94af783847f43328b7b8e


Created version '1' of model 'housing-price-predictor'.


<ModelVersion: aliases=[], creation_timestamp=1782966397233, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1782966397233, metrics=None, model_id=None, name='housing-price-predictor', params=None, run_id='25c7141d97e94af783847f43328b7b8e', run_link=None, source='models:/m-5f4d554584d04dbd84c7e5c7514fc97c', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>